In [1]:
import torch 
USE_CUDA = torch.cuda.is_available()
device = torch.device('cuda:0' if USE_CUDA else 'cpu')

In [2]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import ffmpeg as ff
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
import pickle
import datetime
from facenet_pytorch import MTCNN
import logging
from einops import rearrange
import logging

from torchvision import transforms, utils, models
from PIL import Image
# import face_recognition

In [ ]:
import logging

logging.basicConfig(
    filename='./LOG/random_user.log',  
    filemode='a',
    # level=logging.DEBUG,
    format='[%(asctime)s]::%(process)d::%(levelname)s::%(message)s'
)
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)


In [4]:
number_of_samples = 120

In [5]:
def get_number_of_frames(file_path: str) -> int:
    probe = ff.probe(file_path)
    video_streams = [stream for stream in probe["streams"] if stream["codec_type"] == "video"]
    #width = video_streams[0]['coded_width']
    #height = video_streams[0]['coded_height']
    del probe
    return video_streams[0]['nb_frames']

In [ ]:
def extract_audio_waveform(file_path: str, sr: int = 16000) -> np.ndarray:
    (
        ff
        .input(file_path)
        .output('pipe:', format='f32le', acodec='pcm_f32le', ac=1, ar=str(sr), loglevel='quiet')
        .run(capture_stdout=True)
    )
    raw = ff.input(file_path).output('pipe:', format='f32le', acodec='pcm_f32le', ac=1, ar=str(sr), loglevel='quiet').run(capture_stdout=True)[0]
    waveform = np.frombuffer(raw, dtype=np.float32) 
    return waveform

In [7]:
#추출 할수 있는 얼굴
def extract_N_video_frames(frame_num: str ,file_path: str, number_of_samples: int = 15) -> List[np.ndarray]:
    full_video_frames = []#프레임 데이터를 저장할 배열 만들기
    frame_num=number_of_samples #추출된 프레임의 개수
    begin_num=0 # 프레임의 시작index
    indexes=[None] * number_of_samples # [0,2.7,5.4......]->[0,2,5......] // [0, 1, 2, 3, 4, 5.... 450?]
    get_frames=int(get_number_of_frames(file_path))
    x=get_frames/number_of_samples
    
    for i in range(frame_num):
        indexes[i]=int(begin_num)
        # print(type(get_number_of_frames(file_path)))
        begin_num+=x
    cap = cv2.VideoCapture(file_path)
    for t_idx, ind in enumerate(indexes):
        i=ind
        cap.set(1, ind)
        res, frame = cap.read()#프레임을 읽습니다
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        full_video_frames.append(frame)#RGB 형식으로 변환하여 video_frames에 추가합니다
    cap.release()
    del cap, indexes
    return full_video_frames


In [8]:
def resize_image(image: np.ndarray, new_size: Tuple[int,int]) -> np.ndarray:
    image = np.array(image)
    return cv2.resize(image, new_size, interpolation = cv2.INTER_AREA)

In [9]:
def extract_third_character(filename):
    return filename[:3]
def reading_label_data(file_name: str, dictionary: Dict[str,str]) -> np.ndarray:#영상의 5대 점수를 획득했습니다
    index = int(extract_third_character(file_name)) 
    gt = dictionary.get(index, "ID not found")
    features = ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']
    extracted_data = [float(gt[label]) for label in features]
    return np.stack(extracted_data).reshape(5,1)

In [10]:
def find_script_by_id(transcriptions_filepath,file_name):
    df = pd.read_csv(filepath_or_buffer=transcriptions_filepath, encoding="utf-8",dtype={'pid': str})
    if "SCENE" in file_name:
        filtered = df[(df['clip_name']+".mp4" == file_name)]
    elif "TEST" in file_name:
        filtered = df[(df['clip_name']+".mp4" == file_name)]
    if not filtered.empty:
        # print(file_name+"----"+filtered['transcript'].values[0])
        # print()
        return filtered['transcript'].values[0]
        
    else:
        logger.debug(file_name)
        logger.debug('ERROR. No Exist transcript')
        return None

In [11]:
def preprocessing_input(file_path: str, file_name: str,transcriptions_filepath:str, dictionary: Dict[str, str]) -> Tuple[
    np.ndarray, np.ndarray, np.ndarray]:
    logger.debug(file_path)
    transcriptions = find_script_by_id(transcriptions_filepath, file_name)
    if transcriptions is None:
        logger.debug("transcriptions is None")
        return (None, None, None, None)
 
    #Audio
    extracted_audio_raw = extract_audio_waveform(file_path=file_path)#오디오 데이터 가져오기
    frame_num = int(get_number_of_frames(file_path))
    # Video
    full_video_frames = extract_N_video_frames(frame_num=frame_num,file_path=file_path, number_of_samples=number_of_samples)#128프레임 획득
    resized_Full_images = [resize_image(image=im, new_size=(224, 224)) for im in full_video_frames]#크기를 조정하다
    if len(full_video_frames)==number_of_samples:
        preprocessed_full_video = np.stack(resized_Full_images)#하나의 배열을 합성하다
        video_gt = reading_label_data(file_name=file_name, dictionary=dictionary)#태그 읽기
        del full_video_frames
        return (extracted_audio_raw,preprocessed_full_video,transcriptions, video_gt)
    else:
        logger.debug("frames not enough")
    return (None,None,None,None)

In [12]:
def reshape_to_expected_fullinput(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    x2_list = []
    x3_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
        x2_list.append(dataset[i][2])
        x3_list.append(dataset[i][3])
    return (x0_list, np.stack(x1_list), np.stack(x2_list), np.stack(x3_list))

### data extraction

In [ ]:
import os, pickle, datetime, gc 
import pandas as pd
from tqdm import tqdm
from multiprocessing import get_context

# 필요한 함수: preprocessing_input, get_number_of_frames
# from your_pkg import preprocessing_input, get_number_of_frames


def _process_one(args):
    """
    단일 샘플 처리: data 또는 None을 반환합니다
    (오류가 발생하거나 조건을 만족하지 못하면 건너뜁니다).
    args: (filePath, gt_dict, transcriptions_filepath)
    """
    filePath, gt_dict, transcriptions_filepath = args
    try:
        if not os.path.isfile(filePath):
            return None

        # 먼저 프레임 수를 빠르게 필터링
        if int(get_number_of_frames(filePath)) <= number_of_samples + 1:
            return None

        filename = os.path.basename(filePath)
        data = preprocessing_input(
            file_path=filePath,
            file_name=filename,
            transcriptions_filepath=transcriptions_filepath,
            dictionary=gt_dict,
        )
        if data is None or data[0] is None:
            return None
        return data
    except Exception as e:
        # 여기에서 로그를 남길 수 있습니다: print나 logger로 filePath와 e를 기록해서 디버깅에 활용하세요.
        # print(f"[WARN] error on {filePath}: {e}")
        return None


if __name__ == "__main__":
    # ============ 설정 ============
    allsavename = "/ssd_data/qx/Personality_Llama/fullshot_rowaudio_label_120frame_keti/train_set.dat"
    train_path = "/ssd_data/qx/KETIdata_by_scenario/speaking_only/divided_by_person/SCENE3/train60s"
    test_path = "/ssd_data/qx/KETIdata_by_scenario/speaking_only/divided_by_person/SCENE3/test60s"
    excel_path = "/ssd_data/qx/KETIdata/keti_corpus_labels.csv"
    transcriptions_filepath = "/ssd_data/qx/KETIdata/by_panel_60s_full_transcripts.csv"

    NUM_WORKERS = max(1, os.cpu_count() - 1)
    CHUNKSIZE = 4

    # ============ 레이블을 읽고 gt 딕셔너리 생성 ============
    df = pd.read_csv(excel_path)
    gt = (
        df[["id", "openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]]
        .set_index("id")[["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]]
        .to_dict(orient="index")
    )

    # ============ 후보 비디오 나열 ============
    exts = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
    train_file_list = [
        os.path.join(train_path, fn) for fn in os.listdir(train_path)
        if os.path.splitext(fn.lower())[1] in exts
    ]
    train_file_list.sort()
    # 인자를 리스트로 패킹 [(filePath, gt, transcriptions_filepath), ...]
    train_tasks = [(fp, gt, transcriptions_filepath) for fp in train_file_list]

    # ============ 병렬 처리 ============
    t1 = datetime.datetime.utcnow()
    training_set_data = []

    # spawn 방식을 강제하지 않고 기본값 사용(리눅스에서는 보통 fork, 윈도우에서는 spawn)
    ctx = get_context()  # 또는 get_context("fork") (Linux/mac에서만 사용)
    with ctx.Pool(processes=NUM_WORKERS) as pool:
        for out in tqdm(
            pool.imap_unordered(_process_one, train_tasks, chunksize=CHUNKSIZE),
            total=len(train_tasks),
            desc="Building train_set (multiprocessing)"
        ):
            if out is not None:
                training_set_data.append(out)
                
    test_file_list = [
        os.path.join(test_path, fn) for fn in os.listdir(test_path)
        if os.path.splitext(fn.lower())[1] in exts
    ]
    test_file_list.sort()
    # 인자를 리스트로 패킹 [(filePath, gt, transcriptions_filepath), ...]
    test_tasks = [(fp, gt, transcriptions_filepath) for fp in test_file_list]

    # ============ 병렬 처리 ============
    t1 = datetime.datetime.utcnow()

    # spawn 방식을 강제하지 않고 기본값 사용(리눅스에서는 보통 fork, 윈도우에서는 spawn)
    ctx = get_context()  # 또는 get_context("fork") (Linux/mac에서만 사용)
    with ctx.Pool(processes=NUM_WORKERS) as pool:
        for out in tqdm(
            pool.imap_unordered(_process_one, test_tasks, chunksize=CHUNKSIZE),
            total=len(test_tasks),
            desc="Building test_set (multiprocessing)"
        ):
            if out is not None:
                training_set_data.append(out)

    # ============ reshape + 저장 ============
    gc.collect()
    training_set_data = reshape_to_expected_fullinput(training_set_data)

    with open(allsavename, "wb") as f:
        pickle.dump(training_set_data, f, protocol=4)

    t2 = datetime.datetime.utcnow()
    print("Elapsed time: " + str(t2 - t1))
    gc.collect()


Building train_set (multiprocessing):   0%|          | 0/2143 [00:00<?, ?it/s]

Building test_set (multiprocessing): 100%|██████████| 762/762 [06:15<00:00,  2.03it/s]


Elapsed time: 0:08:28.826700


In [ ]:
import os, pickle, datetime, gc
import pandas as pd
from tqdm import tqdm
from multiprocessing import get_context

# 이미 보유한 함수: preprocessing_input, get_number_of_frames
# from your_pkg import preprocessing_input, get_number_of_frames


def _process_one(args):
    """
    단일 샘플 처리: data 또는 None을 반환합니다
    (오류가 발생하거나 조건을 만족하지 못하면 건너뜁니다).
    args: (filePath, gt_dict, transcriptions_filepath)
    """
    filePath, gt_dict, transcriptions_filepath = args
    try:
        if not os.path.isfile(filePath):
            return None

        # 먼저 프레임 수를 빠르게 필터링
        if int(get_number_of_frames(filePath)) <= number_of_samples + 1:
            return None

        filename = os.path.basename(filePath)
        data = preprocessing_input(
            file_path=filePath,
            file_name=filename,
            transcriptions_filepath=transcriptions_filepath,
            dictionary=gt_dict,
        )
        if data is None or data[0] is None:
            return None
        return data
    except Exception as e:
        # 여기에서 로그를 남길 수 있습니다: print 또는 logger로 filePath와 e를 기록하여 디버깅에 활용
        # print(f"[WARN] error on {filePath}: {e}")
        return None


if __name__ == "__main__":
    # ============ 설정 ============
    allsavename = "/ssd_data/qx/Personality_Llama/fullshot_rowaudio_label_120frame_keti/valid_set.dat"
    path = "/ssd_data/qx/KETIdata_by_scenario/speaking_only/divided_by_person/SCENE3/valid60s"
    excel_path = "/ssd_data/qx/KETIdata/keti_corpus_labels.csv"
    transcriptions_filepath = "/ssd_data/qx/KETIdata/by_panel_60s_full_transcripts.csv"

    NUM_WORKERS = max(1, os.cpu_count() - 1)
    CHUNKSIZE = 4

    # ============ 레이블 파일을 읽고 gt 딕셔너리 생성 ============
    df = pd.read_csv(excel_path)
    gt = (
        df[["id", "openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]]
        .set_index("id")[["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]]
        .to_dict(orient="index")
    )

    # ============ 후보 비디오 파일 나열 ============
    exts = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
    file_list = [
        os.path.join(path, fn) for fn in os.listdir(path)
        if os.path.splitext(fn.lower())[1] in exts
    ]
    file_list.sort()

    # 인자를 리스트로 묶기 [(filePath, gt, transcriptions_filepath), ...]
    tasks = [(fp, gt, transcriptions_filepath) for fp in file_list]

    # ============ 병렬 처리 ============
    t1 = datetime.datetime.utcnow()
    training_set_data = []

    # spawn 방식을 강제하지 않고 기본값 사용 (Linux에서는 일반적으로 fork, Windows에서는 spawn)
    ctx = get_context()  # 또는 get_context("fork") (Linux/mac에서만 사용)
    with ctx.Pool(processes=NUM_WORKERS) as pool:
        for out in tqdm(
            pool.imap_unordered(_process_one, tasks, chunksize=CHUNKSIZE),
            total=len(tasks),
            desc="Building valid_set (multiprocessing)"
        ):
            if out is not None:
                training_set_data.append(out)

    # ============ reshape 후 저장 ============
    gc.collect()
    training_set_data = reshape_to_expected_fullinput(training_set_data)

    with open(allsavename, "wb") as f:
        pickle.dump(training_set_data, f, protocol=4)

    t2 = datetime.datetime.utcnow()
    print("Elapsed time: " + str(t2 - t1))
    gc.collect()


Building valid_set (multiprocessing): 100%|██████████| 698/698 [06:05<00:00,  1.91it/s]


Elapsed time: 0:06:33.424249
